In [6]:
import z3
import numpy as np
from z3 import (
    DeclareSort, 
    SetSort, 
    Const, 
    BoolSort, 
    IntSort,
    IsSubset, 
    Union, 
    Intersect, 
    SetDifference, 
    EmptySet,
    Cardinality, 
    ForAll, 
    Implies, 
    And, 
    Or, 
    Not, 
    IntVal, 
    BoolVal, 
    Solver, 
    sat
)

ImportError: cannot import name 'Cardinality' from 'z3' (/home/fausto/mambaforge/envs/pLoT_RSA/lib/python3.12/site-packages/z3/__init__.py)

## Tiny s-expression parser (just enough for the given grammar)

In [ ]:

def tokenize(s: str):
    s = s.replace("(", " ( ").replace(")", " ) ").replace("|", " | ")
    return [tok for tok in s.split() if tok]

def parse(tokens):
    """Parses a single s-expression or atom."""
    tok = tokens.pop(0)
    if tok == '(':
        lst = []
        while tokens[0] != ')':
            lst.append(parse(tokens))
        tokens.pop(0)  # ')'
        return tuple(lst)  # ('op', arg1, arg2, ...)
    elif tok == '|':
        return '|'
    else:
        return tok

def parse_production(s: str):
    """Returns (composition_ast, [q2_ast, q3_ast, q4_ast])"""
    raw = tokenize(s)
    # Split by top-level pipes (we keep it simple by re-tokenizing slices)
    parts = []
    buf = []
    depth = 0
    for t in raw:
        if t == '(':
            depth += 1
            buf.append(t)
        elif t == ')':
            depth -= 1
            buf.append(t)
        elif t == '|' and depth == 0:
            parts.append(' '.join(buf).strip())
            buf = []
        else:
            buf.append(t)
    if buf:
        parts.append(' '.join(buf).strip())
    if len(parts) != 4:
        raise ValueError(f"Expected 4 pipe-separated parts, got {len(parts)}")
    comp_ast = parse(tokenize(parts[0]))
    q_asts   = [parse(tokenize(p)) for p in parts[1:]]
    return comp_ast, q_asts



## Extract (LeftArg, RightArg) from the composition

In [ ]:

def extract_args_from_composition(comp_ast):
    """
    Expect shape: (( X.Q <LeftArg> ) <RightArg>)
    We return the ASTs for LeftArg and RightArg.
    """
    # comp_ast should be ('(', ('(', 'X.Q', LeftArg), RightArg)   in our tuple form
    if not isinstance(comp_ast, tuple) or len(comp_ast) != 2:
        # could also be eta-expanded or with a superfluous lambda; accept a leading lambda-ish
        # Try to peel a λx. (…) wrapper if present: ('λx', inner)
        if isinstance(comp_ast, tuple) and len(comp_ast) == 1 and isinstance(comp_ast[0], tuple):
            comp_ast = comp_ast[0]
    # Now expect ((X.Q <L>) <R>)
    if not (isinstance(comp_ast, tuple) and len(comp_ast) == 2):
        raise ValueError("Composition not of the form ((X.Q <A>) <B>)")
    inner, right = comp_ast
    if not (isinstance(inner, tuple) and len(inner) == 2):
        raise ValueError("Inner application not of the form (X.Q <A>)")
    head, left = inner
    if head != 'X.Q':
        # Allow possible tuple ('X.Q',) when parsed oddly
        if not (isinstance(head, str) and head == 'X.Q'):
            raise ValueError("Expected X.Q at the head position of composition")
    return left, right



## 3) AST utilities: substitute, pretty-print


In [ ]:

def ast_substitute(ast, mapping):
    """Capture-avoiding substitute of atoms using `mapping` (str -> AST)."""
    if isinstance(ast, tuple):
        return tuple(ast_substitute(x, mapping) for x in ast)
    if isinstance(ast, str) and ast in mapping:
        return mapping[ast]
    return ast

def ast_to_string(ast):
    if isinstance(ast, tuple):
        return "(" + " ".join(ast_to_string(x) for x in ast) + ")"
    return ast



## 4) z3 translation


In [9]:

# We treat individuals as an uninterpreted sort E and sets as Set[E].
E = DeclareSort('E')
SetE = SetSort(E)

# Base variables (the *unpacked* quantifiers are interpreted over these):
L = Const('L', SetE)  # the primitive X.L
R = Const('R', SetE)  # the primitive X.R
C = Const('C', SetE)  # X.c  (universe/context)

In [ ]:
s = z3.Solver()
s.add(IsSubset(L, R))
s.add(z3.Not(IsSubset(C, R)))


solv = s.check()
print(s.model())

Z3Exception: b'set-has-size is not supported'

In [ ]:

# We treat individuals as an uninterpreted sort E and sets as Set[E].
E = DeclareSort('E')
SetE = SetSort(E)

# Base variables (the *unpacked* quantifiers are interpreted over these):
L = Const('L', SetE)  # the primitive X.L
R = Const('R', SetE)  # the primitive X.R
C = Const('C', SetE)  # X.c  (universe/context)

def to_z3(ast):
    """
    Translate AST to a z3 Bool/Int/Set[E] expression.
    We rely on the operators used in your CFG.
    """
    if isinstance(ast, str):
        # atoms
        if ast in ('true', 't', 'True'):
            return BoolVal(True)
        if ast in ('false', 'f', 'False'):
            return BoolVal(False)
        if ast == '0':
            return IntVal(0)
        if ast == '1':
            return IntVal(1)
        if ast == 'X.L':
            return L
        if ast == 'X.R':
            return R
        if ast == 'X.c':
            return C
        # passthrough unknown symbols (shouldn’t occur in finished formulas)
        return ast

    # compound
    if not ast:
        raise ValueError("Empty tuple AST")
    head, *args = ast
    # Set-valued expressions
    if head == 'universe':
        # (universe X.c) -> C
        return to_z3(args[0])  # should be C
    if head == 'union':
        a, b = map(to_z3, args)
        return Union(a, b)
    if head == 'intersection':
        a, b = map(to_z3, args)
        return Intersect(a, b)
    if head == 'setminus':
        a, b = map(to_z3, args)
        return SetDifference(a, b)

    # Int-valued
    if head == 'cardinality':
        # (cardinality <set> <context>) -> Card(set)
        s, _ctx = args
        return Cardinality(to_z3(s))
    if head == '+':
        a, b = map(to_z3, args)
        return a + b
    if head == '-':
        a, b = map(to_z3, args)
        return a - b

    # Bool-valued
    if head == 'intEq':
        a, b = map(to_z3, args)
        return a == b
    if head == 'intGt':
        a, b = map(to_z3, args)
        return a > b
    if head == 'not':
        (a,) = args
        return Not(to_z3(a))
    if head == 'and':
        a, b = map(to_z3, args)
        return And(a, b)
    if head == 'or':
        a, b = map(to_z3, args)
        return Or(a, b)

    # Application nodes (should not appear post-unpacking)
    if head == 'X.Q':
        raise ValueError("Unexpected X.Q after unpacking")

    raise ValueError(f"Unknown head: {head}")



## 5) Public API


In [ ]:

def unpack_and_translate(production: str, assume_subsets=True):
    """
    Input: a 4-part production string.
    Output: dict with:
      - 'left_arg', 'right_arg'        : ASTs used by composition
      - 'unpacked_asts' (list of 3)    : each quantifier with X.L/X.R substituted
      - 'unpacked_strs' (list of 3)    : pretty-printed s-exprs
      - 'z3' (list of 3)               : z3 Bool expressions over L, R, C
      - 'z3_assumptions'               : optional [L ⊆ C, R ⊆ C]
    """
    comp_ast, q_asts = parse_production(production)
    left_arg, right_arg = extract_args_from_composition(comp_ast)

    # Substitution map: in quantifier bodies, X.L -> left_arg, X.R -> right_arg
    mapping = {'X.L': left_arg, 'X.R': right_arg}

    unpacked_asts = [ast_substitute(q, mapping) for q in q_asts]
    unpacked_strs = [ast_to_string(q) for q in unpacked_asts]
    z3_exprs = [to_z3(q) for q in unpacked_asts]

    assumptions = []
    if assume_subsets:
        assumptions = [IsSubset(L, C), IsSubset(R, C)]

    return {
        'left_arg': left_arg,
        'right_arg': right_arg,
        'unpacked_asts': unpacked_asts,
        'unpacked_strs': unpacked_strs,
        'z3': z3_exprs,
        'z3_assumptions': assumptions,
        'z3_vars': {'L': L, 'R': R, 'C': C, 'SetE': SetE, 'E': E}
    }

def equivalent(production_a: str, production_b: str, which: int, assume_subsets=True):
    """
    Check semantic equivalence of the 'which'-th quantifier (2, 3, or 4) in two productions:
    ∀L,R,C. (assumptions) ⇒ (φ_a ↔ φ_b)
    Returns: ('unsat' means equivalent, 'sat' means counterexample exists) and optional model.
    """
    if which not in (2,3,4):
        raise ValueError("Argument 'which' must be 2, 3, or 4")

    pa = unpack_and_translate(production_a, assume_subsets=assume_subsets)
    pb = unpack_and_translate(production_b, assume_subsets=assume_subsets)

    fa = pa['z3'][which-2]
    fb = pb['z3'][which-2]
    asm = pa['z3_assumptions']  # same vars shared

    s = Solver()
    # Negate the equivalence to look for a counterexample:
    # assumptions ∧ (fa ⊕ fb)
    s.add(And(*(asm + [fa != fb])) if asm else (fa != fb))
    res = s.check()
    if res == sat:
        return 'sat', s.model()   # counterexample found: NOT equivalent
    else:
        return 'unsat', None      # no counterexample: equivalent (under given theory)



## 6) Minimal demo (use your own strings at call site)


In [ ]:

if __name__ == "__main__":
    prod = """( ( X.Q ( setminus X.R X.L ) ) X.L ) |
              ( intEq 1 ( cardinality ( intersection ( intersection ( intersection ( setminus X.R X.L ) X.L ) ( universe X.c ) ) X.L ) X.c ) ) |
              ( intGt ( cardinality X.L X.c ) ( cardinality X.R X.c ) ) |
              ( intEq ( cardinality X.R X.c ) 0 )"""

    out = unpack_and_translate(prod)
    print("LeftArg :", ast_to_string(out['left_arg']))
    print("RightArg:", ast_to_string(out['right_arg']))
    print("\nUNPACKED:")
    for i, s in enumerate(out['unpacked_strs'], start=2):
        print(f"Q{i}:", s)

    # Example: compare Q4 of prod with a hand-simplified version
    prod_simpler = """( ( X.Q ( setminus X.R X.L ) ) X.L ) |
                      ( intEq ( cardinality ( intersection ( setminus X.R X.L ) X.L ) X.c ) 1 ) |
                      ( intGt ( cardinality X.L X.c ) ( cardinality X.R X.c ) ) |
                      ( intEq ( cardinality X.L X.c ) 0 )"""
    res, mdl = equivalent(prod, prod_simpler, which=4)
    print("\nQ4 equivalence:", res)
    if mdl:
        print("Counterexample model:", mdl)
